# Ensemble sin Leakage — nueva_info

**Problema con el flujo anterior:**
- Los modelos del profe se entrenaron con **todo el 70%** (df_70).
- `X_va` (10% del 70%) fue visto en entrenamiento → pesos del ensemble ajustados sobre predicciones in-sample → **overfitting**.
- El profe posiblemente evaluó/seleccionó modelos sobre el 30% (df_30) → **selection bias**.

**Solución implementada:**
1. Modelos predicen sobre **todo el 30%** (datos que nunca vieron en entrenamiento → out-of-sample genuino).
2. Partimos las ventanas del 30% en **dos mitades**:
   - `ens_fit` (50%) → para optimizar pesos del ensemble
   - `held_out` (50%) → evaluación final que **nadie** tocó jamás
3. Los pesos del ensemble se aprenden sobre datos limpios, y la evaluación es sobre datos completamente vírgenes.

$$\text{df\_70 (100\%)} \xrightarrow{\text{profe}} \text{Modelos entrenados}$$
$$\text{df\_30} \xrightarrow{\text{predict}} \begin{cases} \text{50\%} \to \text{fit ensemble weights} \\ \text{50\%} \to \text{held-out final eval} \end{cases}$$

In [ ]:
import os
import sys
import importlib

# ---- Colab: clonar repo y subir nueva_info/ ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('heartrate-forecasting'):
        !git clone https://github.com/AllanDBB/heartrate-forecasting.git
    os.chdir('heartrate-forecasting')
    # Subir nueva_info/ manualmente (no esta en git)
    if not os.path.exists('nueva_info/tcn.keras'):
        from google.colab import files
        print('⚠️  Sube la carpeta nueva_info/ (6 .keras + 2 .csv)')
        print('   Puedes arrastrarla al panel de archivos, o usar:')
        print('   !gdown --folder <ID_DRIVE> -O nueva_info/')
        raise FileNotFoundError('nueva_info/ no encontrada. Subela y re-ejecuta esta celda.')

REPO_DIR = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')) if not IN_COLAB else os.getcwd()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('REPO_DIR:', REPO_DIR)
print('Colab:', IN_COLAB)

import main
import utils
importlib.reload(utils)
importlib.reload(main)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Verificación de particiones y semilla

In [ ]:
# Cargar particiones del profe (nueva_info)
df70_nueva = utils.read_split_dataframe('nueva_info/df_70.csv')
df30_nueva = utils.read_split_dataframe('nueva_info/df_30.csv')

print('df_70:', df70_nueva.shape)
print('df_30:', df30_nueva.shape)

In [ ]:
# Verificar semilla: _legacy_selectRandomColumns(seed=42) reproduce el split
df_full = utils.loadAllFiles('dataset')
print('Dataset completo:', df_full.shape)

df70_regen, df30_regen = utils._legacy_selectRandomColumns(df_full, seed=42)

match_70 = (set(df70_regen.columns) == set(df70_nueva.columns)
            and np.allclose(df70_regen[df70_nueva.columns].values,
                            df70_nueva.values, equal_nan=True))
match_30 = (set(df30_regen.columns) == set(df30_nueva.columns)
            and np.allclose(df30_regen[df30_nueva.columns].values,
                            df30_nueva.values, equal_nan=True))

print('Semilla 42 reproduce df_70?', match_70)
print('Semilla 42 reproduce df_30?', match_30)
print('Columnas solapadas:', sorted(set(df70_regen.columns) & set(df30_regen.columns)))

## 2. Preparar datasets con particion del 30%

In [ ]:
INPUT_SIZE      = 200
OUTPUT_SIZE     = 200
CACHE_DIR       = 'cache_nueva_info'
ENSEMBLE_SEED   = 123   # semilla diferente para el split del ensemble

# ---- Cargar y estandarizar ----
main.ensure_dir(CACHE_DIR)

df_70, df_30, split_meta = main.load_split_dataframes(
    dataset_dir='dataset',
    split_seed=42,
    split_70_path='nueva_info/df_70.csv',
    split_30_path='nueva_info/df_30.csv',
)
df_70, df_30, overlap = utils.sanitize_split_dataframes(df_70, df_30)
print('Overlap eliminado:', overlap)

path_est_70 = os.path.join(CACHE_DIR, 'values_deses_70.csv')
path_est_30 = os.path.join(CACHE_DIR, 'values_deses_30.csv')
df_scaled_70, params_70 = utils.estandarizar(df_70, path_est_70)
df_scaled_30, params_30 = utils.estandarizar(df_30, path_est_30)

print(f'\ndf_70 estandarizado: {df_scaled_70.shape}  ({len(df_70.columns)} series)')
print(f'df_30 estandarizado: {df_scaled_30.shape}  ({len(df_30.columns)} series)')

In [ ]:
# ---- Generar ventanas supervisadas ----
X_30, y_30, ids_30 = utils.series_to_supervised_matrix(
    df_scaled_30, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)
print(f'Ventanas totales del 30%: X={X_30.shape}, y={y_30.shape}')

# ---- Partir ventanas del 30% en dos: ens_fit (50%) y held_out (50%) ----
# Usamos stratify por ids para mantener proporcion de sujetos en ambos lados
(
    X_ens_fit, X_held_out,
    y_ens_fit, y_held_out,
    ids_ens_fit, ids_held_out
) = train_test_split(
    X_30, y_30, ids_30,
    test_size=0.5,
    random_state=ENSEMBLE_SEED,
    stratify=ids_30,   # misma proporcion de sujetos
)

print(f'\n--- Split del 30% ---')
print(f'ens_fit  (ajustar ensemble): X={X_ens_fit.shape}, y={y_ens_fit.shape}')
print(f'held_out (eval final):       X={X_held_out.shape}, y={y_held_out.shape}')
print(f'Sujetos en ens_fit:  {len(set(ids_ens_fit))}')
print(f'Sujetos en held_out: {len(set(ids_held_out))}')

## 3. Verificar parametros de desestandarizacion

Se guardan media y desviacion por serie. Para volver a escala original:
$$\hat{y}_{\text{original}} = \hat{y}_{\text{std}} \times \sigma + \mu$$

In [ ]:
print('params_70 (para restaurar df_70):')
print(params_70.head())
print(f'\nparams_30 (para restaurar df_30):')
print(params_30.head())

# Validar que deses == desestandarizar_ventanas
sample = y_30[:3]
sample_ids = ids_30[:3]
out1 = utils.deses(sample, sample_ids, params_30)
out2 = utils.desestandarizar_ventanas(sample, sample_ids, params_30)
print(f'\ndeses == desestandarizar_ventanas? {np.allclose(out1, out2)}')

## 4. Inferencia individual (modelos de nueva_info)

In [ ]:
from wrappers.KerasPretrainedWrapper import KerasPretrainedWrapper

MODEL_SPECS = {
    'TCN':          'nueva_info/tcn.keras',
    'NBEATS':       'nueva_info/nbeats.keras',
    'LSTM':         'nueva_info/lstm.keras',
    'TiDE':         'nueva_info/tide.keras',
    'EncDec':       'nueva_info/encDec.keras',
    'iTransformer': 'nueva_info/itrans.keras',
}

print('Modelos a cargar:')
for name, path in MODEL_SPECS.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {name:15s} -> {path} ({size_mb:.1f} MB) {"OK" if exists else "MISSING"}')

In [ ]:
# Predecir sobre las DOS mitades del 30%
preds_ens_fit  = {}  # predicciones sobre ens_fit
preds_held_out = {}  # predicciones sobre held_out

for name, model_path in MODEL_SPECS.items():
    try:
        wrapper = KerasPretrainedWrapper(model_path, batch_size=32, name=name).load()
        preds_ens_fit[name]  = wrapper.predict(X_ens_fit)
        preds_held_out[name] = wrapper.predict(X_held_out)
        print(f'[OK] {name}: ens_fit={preds_ens_fit[name].shape}, held_out={preds_held_out[name].shape}')
    except Exception as exc:
        print(f'[SKIP] {name}: {exc}')

print(f'\nModelos cargados: {len(preds_ens_fit)} / {len(MODEL_SPECS)}')

## 5. Desestandarizar y metricas individuales

In [ ]:
# ---- Restaurar escala original ----
y_ens_fit_orig  = utils.desestandarizar_ventanas(y_ens_fit, ids_ens_fit, params_30)
y_held_out_orig = utils.desestandarizar_ventanas(y_held_out, ids_held_out, params_30)

preds_ens_fit_orig  = {n: utils.desestandarizar_ventanas(p, ids_ens_fit, params_30)
                       for n, p in preds_ens_fit.items()}
preds_held_out_orig = {n: utils.desestandarizar_ventanas(p, ids_held_out, params_30)
                       for n, p in preds_held_out.items()}

# ---- Metricas individuales sobre HELD-OUT (nunca visto) ----
print('=== Metricas individuales sobre held_out (50% del 30%) ===')
individual_results_held_out = {}
for name, pred in preds_held_out_orig.items():
    individual_results_held_out[name] = utils.evaluate_all_metrics(y_held_out_orig, pred)

df_ind_held = pd.DataFrame(individual_results_held_out).T
df_ind_held.index.name = 'Modelo'
if 'MAPE' in df_ind_held.columns:
    df_ind_held = df_ind_held.sort_values('MAPE')
df_ind_held

In [ ]:
# ---- Metricas individuales sobre ens_fit (para referencia) ----
print('=== Metricas individuales sobre ens_fit (50% del 30%) ===')
individual_results_fit = {}
for name, pred in preds_ens_fit_orig.items():
    individual_results_fit[name] = utils.evaluate_all_metrics(y_ens_fit_orig, pred)

df_ind_fit = pd.DataFrame(individual_results_fit).T
df_ind_fit.index.name = 'Modelo'
if 'MAPE' in df_ind_fit.columns:
    df_ind_fit = df_ind_fit.sort_values('MAPE')
df_ind_fit

## 6. Ensemble (sin leakage)

- **fit_weights** se ajusta sobre `ens_fit` (50% del 30%, out-of-sample genuino)
- **evaluate** se hace sobre `held_out` (50% del 30%, **nunca visto** por nadie)

In [ ]:
from wrappers.EnsembleWrapper import EnsembleWrapper

ens = EnsembleWrapper()

# split='fit' -> predicciones sobre ens_fit (para aprender pesos)
for name, pred in preds_ens_fit_orig.items():
    ens.add_model(name, pred, split='fit')

# split='eval' -> predicciones sobre held_out (para evaluacion final)
for name, pred in preds_held_out_orig.items():
    ens.add_model(name, pred, split='eval')

ensemble_results = ens.compare_methods(
    y_true_eval=y_held_out_orig,   # solo para evaluar, NUNCA para fit
    y_true_fit=y_ens_fit_orig,     # aqui se aprenden los pesos
    objective_metric='MAPE',
    active_method='optimize',
)

df_ens = pd.DataFrame(ensemble_results).T
df_ens.index.name = 'Metodo Ensemble'
if 'MAPE' in df_ens.columns:
    df_ens = df_ens.sort_values('MAPE')
df_ens

## 7. Tabla final consolidada

Todas las metricas evaluadas sobre **held_out** (datos que nunca participaron en ningun ajuste).

In [ ]:
all_results = {**individual_results_held_out, **ensemble_results}
df_final = pd.DataFrame(all_results).T
df_final.index.name = 'Modelo / Metodo'
if 'MAPE' in df_final.columns:
    df_final = df_final.sort_values('MAPE')

print('=== TABLA FINAL evaluada sobre held_out (nunca visto) ===')
df_final

In [ ]:
# Guardar resultados
csv_path = os.path.join(CACHE_DIR, 'final_results_no_leakage.csv')
df_final.to_csv(csv_path)
print(f'Guardado en: {csv_path}')

## 8. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

utils.plot_metrics_comparison(all_results, title='Comparacion — held_out (sin leakage)')

In [ ]:
y_ens_pred = ens.predict()

utils.plot_forecast_samples(
    y_held_out_orig, y_ens_pred, n_samples=6,
    title='Ensemble Optimizado vs Real (held_out)'
)

In [ ]:
utils.plot_error_over_horizon(
    y_held_out_orig, y_ens_pred,
    title='Ensemble: Error segun horizonte (held_out)'
)

In [ ]:
utils.plot_subject_performance(
    y_held_out_orig, y_ens_pred, ids_held_out,
    title='Ensemble: Rendimiento por sujeto (held_out)'
)

## 9. Diagnostico de overfitting del ensemble

Si el ensemble funciona mucho mejor en `ens_fit` que en `held_out`, hay overfitting en los pesos.

In [ ]:
# Evaluar el ensemble (con los mismos pesos) sobre ens_fit
y_ens_on_fit = ens.predict(model_predictions=preds_ens_fit_orig)
metrics_on_fit = utils.evaluate_all_metrics(y_ens_fit_orig, y_ens_on_fit)

# Evaluar sobre held_out
metrics_on_held = utils.evaluate_all_metrics(y_held_out_orig, y_ens_pred)

diag = pd.DataFrame({
    'ens_fit (pesos aprendidos aqui)': metrics_on_fit,
    'held_out (nunca visto)': metrics_on_held,
})

print('=== Diagnostico de overfitting ===')
print('Si MAPE en ens_fit << held_out -> overfitting en pesos del ensemble')
print('Si son similares -> los pesos generalizan bien')
print()
diag